# Simulação de Cavidade Elíptica para NV-Maser

Este notebook tem como objetivo modelar e simular as propriedades eletromagnéticas de uma cavidade ressonante elíptica preenchida com dielétrico (como safira), focada em aplicações de NV-Maser. 

O objetivo principal é encontrar as condições físicas (dimensões e permissividade) para que ocorra a **degenerescência** entre os modos $TE_{111}$ e $TM_{010}$ exatamente na frequência alvo de transição do centro NV do diamante ($2.87$ GHz).

### Passo 1: Importação de Bibliotecas e Constantes
Iniciamos importando as ferramentas matemáticas e gráficas necessárias, além de definir as constantes físicas e raízes das funções de Bessel.

* $c_0 = 3 \times 10^{11}$ mm/s (Velocidade da luz no vácuo em mm/s)
* $X_{01} = 2.40483$ (Primeira raiz de $J_0(x)$, dita o corte TM)
* $X'_{11} = 1.84118$ (Primeira raiz da derivada de $J_1(x)$, dita o corte TE)
* $f_{alvo} = 2.87$ GHz (Frequência do centro NV)

In [7]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import scipy.constants as sc
import scipy.special as sp
import scipy.optimize as opt
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── constants ─────────────────────────────────────────────────
C0 = 3e11             # Velocidade da luz (mm/s)
TARGET = 2.87         # Frequência alvo em GHz

### Passo 2: Geometria e Frequências de Ressonância

Para uma cavidade elíptica com semi-eixo maior $a$ e excentricidade $e$, definimos um **raio efetivo** $R_{eff}$ para aproximar o comportamento cilíndrico:
$$R_{eff} = a(1 - e^2)^{1/4}$$

A frequência do modo **TM$_{010}$** independe do comprimento da cavidade ($L$) e é dada por:
$$f_{TM_{010}} = \frac{c_0 X_{01}}{2\pi R_{eff} \sqrt{\varepsilon_r}}$$

Para o modo **TE$_{111}$**, a frequência depende do comprimento $L$. Calculamos primeiro a frequência de corte $f_c$ e o termo longitudinal $f_z$:
$$f_{c} = \frac{c_0 X'_{11}}{2\pi R_{eff} \sqrt{\varepsilon_r}}$$
$$f_z = \frac{c_0}{2L\sqrt{\varepsilon_r}}$$
A frequência de ressonância final é:
$$f_{TE_{111}} = \sqrt{f_c^2 + f_z^2}$$

In [8]:
# ── physics (Mathieu Functions) ───────────────────────────────
def kc_mathieu(a, e, m, tipo):
    """ Encontra o autovalor k_c exato usando funções de Mathieu. """
    f = a * e
    xi_0 = np.arccosh(1.0 / e)
    
    def eq(kc):
        q = (kc * f / 2)**2
        val, der = sp.mathieu_modcem1(m, q, xi_0)
        return val if tipo == 'TM' else der

    # Chute inicial baseado na aproximação cilíndrica do raio efetivo
    Reff = a * np.power(1 - e**2, 0.25)
    guess = (2.40483 if tipo == 'TM' else 1.84118) / Reff
    
    # Vetor de varredura para isolar a raiz
    kcs = np.linspace(guess * 0.4, guess * 1.6, 120)
    valores = [eq(k) for k in kcs]
    mudancas = np.where(np.diff(np.sign(valores)))[0]
    
    if len(mudancas) > 0:
        idx = mudancas[0]
        try:
            return opt.brentq(eq, kcs[idx], kcs[idx+1])
        except:
            return guess
    return guess

def fTM(a, e, er):
    kc = kc_mathieu(a, e, 0, 'TM')
    return (C0 * kc / (2 * np.pi * np.sqrt(er))) / 1e9

def fTEcutoff(a, e, er):
    kc = kc_mathieu(a, e, 1, 'TE')
    return (C0 * kc / (2 * np.pi * np.sqrt(er))) / 1e9

def fTE(a, e, L, er):
    fc = fTEcutoff(a, e, er)
    fz = C0 / (2 * L * np.sqrt(er)) / 1e9
    return np.sqrt(fc**2 + fz**2)

def mode_volume(a, e, L):
    """ Volume da cavidade elíptica em mm³ """
    return np.pi * a * a * np.sqrt(1 - e**2) * L

### Passo 3: Busca pela Degenerescência e Volume Modal

Para que o Maser funcione otimamente, os modos $TE_{111}$ e $TM_{010}$ devem ser **degenerados** (terem a mesma frequência). Como o modo TM é fixo e o modo TE depende de $L$, usamos um método de **Bisseção** para varrer o espaço e encontrar o comprimento $L_{degen}$ exato onde as curvas se cruzam.

Também calculamos o volume do modo:
$$V = \pi a^2 \sqrt{1 - e^2} L$$

In [9]:
# ── fields (Mathieu Distribution) ─────────────────────────────
def compute_mathieu_field_unmasked(X, Y, a, e, m, tipo):
    """ Calcula o campo puro em todo o grid cartesiano, sem aplicar máscara. """
    f = a * e
    kc = kc_mathieu(a, e, m, tipo)
    q = (kc * f / 2)**2
    
    # Coordenadas elípticas a partir do plano complexo
    Z_complex = (X + 1j * Y) / f
    xi_eta = np.arccosh(Z_complex + 1e-15j) # Evita singularidade focal
    xi, eta = np.real(xi_eta), np.imag(xi_eta)
    
    # Vetorização rigorosa
    v_cem = np.vectorize(lambda ang: sp.mathieu_cem(m, q, ang * 180.0 / np.pi)[0])
    v_mod = np.vectorize(lambda rad: sp.mathieu_modcem1(m, q, rad)[0])
    
    return np.real(v_mod(xi) * v_cem(eta))

def apply_ellipse_mask(X, Y, a, e, campo):
    """ Recorta o grid limitando a visualização à parede física da cavidade. """
    mask = (X**2 / a**2) + (Y**2 / (a**2 * (1 - e**2))) <= 1.0
    return np.where(mask, campo, np.nan)

def F_Hz(X, Y, a, e):
    campo = compute_mathieu_field_unmasked(X, Y, a, e, 1, 'TE')
    return apply_ellipse_mask(X, Y, a, e, campo)

def F_Eperp(X, Y, a, e):
    # O campo elétrico transversal deriva do gradiente magnético não mascarado
    Hz_unmasked = compute_mathieu_field_unmasked(X, Y, a, e, 1, 'TE')
    dy, dx = np.gradient(Hz_unmasked)
    intensidade = np.sqrt(dx**2 + dy**2)
    return apply_ellipse_mask(X, Y, a, e, intensidade)

def F_Hperp(X, Y, a, e):
    # Por aproximação da equação de Helmholtz no TE111
    return F_Eperp(X, Y, a, e)

def F_Ez(X, Y, a, e):
    campo = compute_mathieu_field_unmasked(X, Y, a, e, 0, 'TM')
    return apply_ellipse_mask(X, Y, a, e, campo)

def F_Hphi(X, Y, a, e):
    Ez_unmasked = compute_mathieu_field_unmasked(X, Y, a, e, 0, 'TM')
    dy, dx = np.gradient(Ez_unmasked)
    intensidade = np.sqrt(dx**2 + dy**2)
    return apply_ellipse_mask(X, Y, a, e, intensidade)

### Passo 4: Distribuição dos Campos Eletromagnéticos (Funções de Bessel)

Os campos internos dependem das coordenadas espaciais ($r, \phi$). Eles são modelados usando funções de Bessel de primeiro tipo ($J_n$) e suas derivadas ($J'_n$). 

O campo magnético transversal $|H_\perp|$ do modo $TE_{11}$, vital para o acoplamento de spin no centro NV, depende das contribuições radiais e azimutais derivadas da equação de Helmholtz na geometria elíptica.

In [10]:
# ── optimization & UI helpers ─────────────────────────────────
def Ldegen_bisect(a, e, er, Lo=20.0, Hi=500.0, tol=1e-6):
    ftm = fTM(a, e, er)
    if fTE(a, e, Lo, er) <= ftm or fTE(a, e, Hi, er) >= ftm:
        return None
        
    for _ in range(80):
        mid = (Lo + Hi) / 2
        if fTE(a, e, mid, er) > ftm: Lo = mid
        else: Hi = mid
        if Hi - Lo < tol: break
    return {'L': (Lo + Hi) / 2, 'f': fTE(a, e, (Lo + Hi) / 2, er)}

def _overlay(ax, a, e):
    b = a * np.sqrt(1 - e**2)
    foc = a * e
    th = np.linspace(0, 2 * np.pi, 400)
    ax.plot(a * np.cos(th), b * np.sin(th), 'w-', lw=1, alpha=0.6)
    ax.plot([-foc, foc], [0, 0], 'x', color='cyan', ms=6, mew=2, zorder=5, label='Focos')
    ax.plot(0, 0, '+', color='white', ms=8, zorder=5)

def draw_field(ax, Ffn, a, e, title, cmap='inferno', signed=False, N=150):
    b = a * np.sqrt(1 - e**2)
    x, y = np.linspace(-a * 1.05, a * 1.05, N), np.linspace(-b * 1.05, b * 1.05, N)
    X, Y = np.meshgrid(x, y)
    F = Ffn(X, Y, a, e)
    
    vm = np.nanmax(np.abs(F)) or 1.0
    kw = dict(extent=[-a*1.05, a*1.05, -b*1.05, b*1.05], origin='lower', aspect='equal', cmap=cmap)
    ax.imshow(F, vmin=-vm if signed else 0, vmax=vm, **kw)
    _overlay(ax, a, e)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('x (mm)', fontsize=8)
    ax.set_ylabel('y (mm)', fontsize=8)

def draw_s11(ax, fte, ftm):
    flo = min(fte, ftm, TARGET) * 0.87
    fhi = max(fte, ftm, TARGET) * 1.13
    f   = np.linspace(flo, fhi, 3000)
    DEPTH = -35

    def S11_dip(fc, Q):
        d = 2 * Q * (f - fc) / fc
        return DEPTH / (1 + d**2)

    ax.axhline(0, color='grey', lw=0.8, alpha=0.5)
    ax.plot(f, S11_dip(fte, 9000), color='#1F77B4', lw=2, label=f'TE₁₁₁  {fte:.4f} GHz')
    ax.plot(f, S11_dip(ftm, 7000), color='#D62728', lw=2, label=f'TM₀₁₀  {ftm:.4f} GHz')
    ax.axvline(TARGET, color='#2CA02C', ls='--', lw=1.5, label=f'{TARGET} GHz')

    df = abs(fte - ftm) * 1000
    ax.annotate(f'|Δf| = {df:.3f} MHz', xy=(0.98, 0.08), xycoords='axes fraction',
                ha='right', va='bottom', fontsize=8, color='purple',
                bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', ec='purple', alpha=0.85))

    ax.set_ylim(DEPTH * 1.15, 5)
    ax.set_ylabel('S₁₁ (dB)', fontsize=8)
    ax.set_xlabel('Frequência (GHz)', fontsize=8)
    ax.set_title('S₁₁ Simulado (Mergulho Lorentziano)', fontsize=9)
    ax.legend(fontsize=7, loc='lower center')
    ax.grid(alpha=0.3)
    ax.tick_params(labelsize=7)

def draw_Lscan(ax, a, e, er, Lcur):
    LSTART, Lmax = 20.0, max(150.0, Lcur * 2.5)
    Ls = np.linspace(LSTART, Lmax, 200)
    fTEs = [fTE(a, e, l, er) for l in Ls]
    ftmC = fTM(a, e, er)
    cross = Ldegen_bisect(a, e, er, Lo=LSTART, Hi=Lmax)

    ax.plot(Ls, fTEs, color='#1F77B4', lw=2, label='TE₁₁₁')
    ax.axhline(ftmC, color='#D62728', lw=2, label=f'TM₀₁₀ = {ftmC:.4f} GHz')
    ax.axhline(TARGET, color='#2CA02C', ls='--', lw=1.5, label='Alvo 2.87 GHz')

    if cross:
        Lc, fc = cross['L'], cross['f']
        ax.plot(Lc, fc, 'o', color='#27AE60', ms=10, label=f'L_degen = {Lc:.2f} mm')
        ax.annotate(f'✓ {Lc:.2f} mm', xy=(Lc, fc), xytext=(Lc+(Lmax-LSTART)*0.03, fc+0.1), arrowprops=dict(arrowstyle='->'))

    ax.axvline(Lcur, color='orange', ls=':', lw=2, label=f'L atual = {Lcur:.1f} mm')
    ax.set_xlabel('Altura da Cavidade L (mm)', fontsize=8)
    ax.set_ylabel('Frequência (GHz)', fontsize=8)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

### Passo 5: Visualização de Dados (Gráficos e Parâmetro S11)

Nesta seção, definimos as funções que vão gerar as visualizações em Matplotlib.
* **Campos 2D**: Exibem a intensidade dos modos dentro da máscara elíptica.
* **Parâmetro $S_{11}$**: Simula a perda de retorno (reflexão) usando um modelo Lorentziano em que a cavidade ressoa produzindo um mergulho ("dip") na reflexão, mostrando a proximidade das frequências $TE$ e $TM$.
* **Varredura em L (L-Scan)**: Mostra visualmente onde ocorre o cruzamento de degenerescência.

In [11]:
# ── master figure ─────────────────────────────────────────────
def build_figure(a, e, L, er, N=150):
    plt.close('all')

    fte  = fTE(a, e, L, er)
    ftm  = fTM(a, e, er)
    df   = abs(fte - ftm) * 1000
    
    vm   = mode_volume(a, e, L) 
    
    onT  = abs(ftm - TARGET) / TARGET < 0.015
    degen = df < 20

    if degen and onT:
        st, sc = f'✓ DEGENERATE @ 2.87 GHz   |Δf| = {df:.3f} MHz', 'green'
    elif degen:
        st, sc = f'~ Degenerate mas fora do alvo   f(TM) = {ftm:.4f} GHz', 'orange'
    else:
        st, sc = f'✗ Not degenerate   |Δf| = {df:.3f} MHz', 'red'

    b = a * np.sqrt(1 - e**2)
    fig = plt.figure(figsize=(18, 11))
    fig.suptitle(
        f'a={a:.2f} mm   b={b:.2f} mm   L={L:.2f} mm   '
        f'e={e:.3f}   ε_r={er:.2f}   Volume={vm/1000:.1f} cm³\n'
        f'f(TE)={fte:.4f} GHz   f(TM)={ftm:.4f} GHz\n{st}',
        fontsize=11, fontweight='bold', color=sc)

    gs = GridSpec(3, 3, figure=fig, hspace=0.55, wspace=0.35)

    draw_field(fig.add_subplot(gs[0, 0]), F_Eperp, a, e, '|E_⊥| TE₁₁ — two-lobe transverse E', N=N)
    draw_field(fig.add_subplot(gs[0, 1]), F_Hz, a, e, 'H_z TE₁₁ — axial H (signed)', cmap='RdBu_r', signed=True, N=N)
    draw_field(fig.add_subplot(gs[0, 2]), F_Hperp, a, e, '|H_⊥| TE₁₁ — NV spin coupling ∝ ∫|H_⊥|²dV', N=N)

    draw_field(fig.add_subplot(gs[1, 0]), F_Ez, a, e, 'E_z TM₀₁ — dome (zero at wall)', N=N)
    draw_field(fig.add_subplot(gs[1, 1]), F_Hphi, a, e, '|H_φ| TM₀₁ — annular ring', N=N)
    draw_s11(fig.add_subplot(gs[1, 2]), fte, ftm)

    draw_Lscan(fig.add_subplot(gs[2, :]), a, e, er, L)

    return fig

### Passo 6: Criação do Painel Principal e UI Interativa

Nesta seção, juntamos todos os gráficos criados no passo anterior usando o `GridSpec` para compor o Dashboard final. 

Em seguida, inicializamos os sliders dinâmicos utilizando `ipywidgets` (caso esteja num ambiente interativo como Jupyter) para manipular parâmetros como Eixo Semimaior ($a$), excentricidade ($e$), permissividade ($\varepsilon_r$), etc., em tempo real.

In [12]:
# ── interface and execution ────────────────────────────────────
def run_interactive():
    style = {'description_width': '190px'}
    lay   = widgets.Layout(width='560px')

    sl = dict(
        er  = widgets.FloatSlider(value=9.4, min=1.0, max=90.0, step=0.05, description='ε_r  (1=ar, 9.4=safira):', style=style, layout=lay),
        a   = widgets.FloatSlider(value=14.2, min=1.0, max=60.0, step=0.02, description='a — semi-eixo maior (mm):', style=style, layout=lay),
        e   = widgets.FloatSlider(value=0.54, min=0.01, max=0.96, step=0.001, description='e — excentricidade:', style=style, layout=lay),
        L   = widgets.FloatSlider(value=26.4, min=1.0, max=150.0, step=0.05, description='L — altura (mm):', style=style, layout=lay),
    )
    out = widgets.Output()

    def update_plot(change=None):
        with out:
            clear_output(wait=True)
            fig = build_figure(sl['a'].value, sl['e'].value, sl['L'].value, sl['er'].value)
            display(fig)
            plt.close(fig)

    for s in sl.values():
        s.observe(update_plot, names='value')

    display(widgets.VBox([
        widgets.HTML('<h3>Dashboard NV-Maser (Modelagem Elíptica Exata via Mathieu)</h3>'),
        *sl.values(),
        out
    ]))
    update_plot()

if __name__ == '__main__':
    run_interactive()